# Testing entanglement with a fixed witness (SDP over the PPT cone)

For a fixed positive-but-not-completely-positive (PnCP) witness `W`, solve

$$\min_\rho\ \mathrm{Tr}(W\rho)\quad\text{s.t.}\quad \rho \succeq 0,\ \ \mathrm{Tr}(\rho)=1,\ \ \rho^{\Gamma}\succeq 0,$$

i.e. minimise `tr(W·ρ)` over **PPT** density matrices `ρ`. Every separable state
obeys `tr(W·σ) ≥ 0`, so a **negative** optimum certifies the minimiser `ρ` as a
PPT **entangled** (bound entangled) state that `W` detects.

This is the convex dual of `detect_trace`: there the state is fixed and the
witness ranges over a finite library; here the witness `W` is fixed and the state
ranges over the whole PPT cone. The SDP is `ppt2.min_ppt_witness`; the at-scale
sweep over a whole form library is `scripts/gen_witness_ppt.jl`.

In [ ]:
include("common.jl")
using Statistics

## Witness library

Load a pool of PnCP witnesses (`pncp_NxM.jld2`, produced by `scripts/gen_pncp.jl`).
Each form is the Choi matrix of a real KSMZ map, read as a complex Hermitian
block-positive witness.

In [ ]:
d = 4
forms = load_batches("../pncp_4x4.jld2")
length(forms)

## The SDP

`min_ppt_witness(W, n, m)` builds the model

```julia
@variable(model, ρ[1:d, 1:d] in HermitianPSDCone())                                  # ρ ⪰ 0, Hermitian
@constraint(model, real(tr(ρ)) == 1)                                                 # unit trace
@constraint(model, Hermitian(partial_transpose(Matrix(ρ), 2, [n, m])) in HermitianPSDCone())  # PPT
@objective(model, Min, real(tr(W * ρ)))
```

and returns `(value, state, detected)`: the optimum `tr(W·ρ)`, the optimising
state `ρ`, and whether `value < -tol`.

In [ ]:
W = forms[3]                       # pick a witness
r = min_ppt_witness(W, d, d)
(value = r.value, detected = r.detected)

## Verify the certified state

The minimiser should be a bona-fide PPT state (`ρ ⪰ 0`, unit trace, `ρ^Γ ⪰ 0`)
whose entanglement is confirmed *independently* of `W` by the level-2 DPS
robustness — a second, unrelated certificate (`> 0` ⟹ entangled).

In [ ]:
ρ = Hermitian(Matrix(r.state))
(ppt            = is_ppt(ρ, d, d),
 eigmin         = eigmin(ρ),
 trace          = real(tr(ρ)),
 tr_Wρ          = real(tr(W * ρ)),          # equals r.value
 dps_robustness = detect_dps(ρ, d, d).value)

## How strong is each witness?

Sweep the first `K` witnesses and record the optimum `tr(W·ρ)` for each: the
witness's PPT-detection strength (more negative = detects a "deeper" bound
entangled state). `gen_witness_ppt.jl` runs this over the whole library in
parallel.

In [ ]:
K = 30
vals = [min_ppt_witness(forms[i], d, d).value for i in 1:K]
(detected = count(<(-1e-8), vals), of = K,
 min = minimum(vals), median = median(vals), max = maximum(vals))

## Feed a witnessed state into the PPT² test

Take the most strongly witnessed state, self-compose it with `ampliation`, and run
all three detection criteria on the composite. The PPT² conjecture predicts the
composite stays separable, so every criterion should land on the separable side.
For a real witness the minimiser is real, so we drop the (numerically zero)
imaginary part before scoring with the real form library.

In [ ]:
best = argmin(vals)
σ = real(Matrix(min_ppt_witness(forms[best], d, d).state))
τ = Hermitian(ampliation(σ, σ, d, d))
(trace      = detect_trace(τ, forms).value,
 ampliation = detect_ampliation(τ, forms, d, d).value,
 dps        = detect_dps(τ, d, d).value)